# Problem set 5 Code

In [42]:
import numpy as np 
import pandas as pd

### Problem 1(c)

In [43]:
sig = 0.036
mu = 0.018
gamma = 10
beta = 0.99

rf = -np.log(beta) + gamma * mu  -0.5 * gamma**2 * sig**2
risk_prem = gamma * sig**2

print(f'Risk Free Rate: {rf}')
print(f'Risk Premium: {risk_prem}')


Risk Free Rate: 0.12525033585350145
Risk Premium: 0.01296


Increasing $\sigma$ increases the risk premium and decreases the risk free rate. Future states are more risky so more demand for risk free which decreases the rate. Increasing $\mu$ increases the risk free rate as increasing mu means we will see increasing consumption in the future. We need a higher risk free rate to have the agent save since they expect higher consumption. Increasing $\gamma$ increases our risk aversion and thus increases the risk premium required to make the agent willing enough to invest. Similarly we are more willing to take a lower risk free rate, when $\gamma\mu < 1/2\gamma^2\sigma^2$. Otherwise the risk free rate increases since our increasing consumption dominates our aversion to risky asset. They won't need a high risk free rate to save the more risk averse they are. Increasing $\beta$ decreases the risk free rate as the agent is more patient and willing to consume less in the current period. Thus, they don't such a high risk free rate to optimally save. 

### Problem 1(d)

In [44]:
epsilon = np.random.normal(0,1,100000)

log_consumption = mu+sig*epsilon

sdf = np.log(beta) - gamma*log_consumption

rf_series = np.full(len(epsilon),rf)

log_ret = rf_series + gamma*sig**2 + sig*epsilon

log_excess_ret = log_ret - rf_series

In [45]:
print('Unconditional Standard Deviations')
print(f'Log consumption growth: {np.round(np.std(log_consumption), 4)}')
print(f'Log SDF:                {np.round(np.std(sdf), 4)}')
print(f'Log risk-free rate:     {np.round(np.std(rf_series), 4)}')
print(f'Log equity return:      {np.round(np.std(log_ret), 4)}')
print(f'Log excess return:      {np.round(np.std(log_excess_ret), 4)}')

from statsmodels.tsa.stattools import acf

ac_log_consumption = acf(log_consumption,nlags=1)[1]
ac_sdf = acf(sdf,nlags=1)[1]
ac_rf = acf(rf_series,nlags=1)[1]
ac_log_ret = acf(log_ret,nlags=1)[1]
ac_excess_ret = acf(log_excess_ret,nlags=1)[1]

print('')
print('First Autocorrelations')
print(f'Log Consumption Growth: {np.round(ac_log_consumption, 4)}')
print(f'Log SDF:                {np.round(ac_sdf, 4)}')
print(f'Risk Free Rate:         {np.round(ac_rf, 4)}')
print(f'Log Equity Return:      {np.round(ac_log_ret, 4)}')
print(f'Log Excess Return:      {np.round(ac_excess_ret, 4)}')


Unconditional Standard Deviations
Log consumption growth: 0.036
Log SDF:                0.3595
Log risk-free rate:     0.0
Log equity return:      0.036
Log excess return:      0.036

First Autocorrelations
Log Consumption Growth: -0.004
Log SDF:                -0.004
Risk Free Rate:         1.0
Log Equity Return:      -0.004
Log Excess Return:      -0.004


In [46]:
print(f'Correlation of consumption growth and equity return {np.corrcoef(log_consumption,log_ret)[0][1]}')

Correlation of consumption growth and equity return 1.0


In [47]:
print(f'expected log excess return {np.mean(log_excess_ret)}')

expected log excess return 0.013143356687638646


These statistics are consistent with the data as our standard deviations are close to the set deviation of 0.036. SDF varies by 10*0.036 which matches.  Risk free rate also is persistent and constant. The autocorrelations are low as consumption doesn't depend on previous period. Since they all depend on log consumption the autocorrelations should be low.

### Problem 2(b)

In [48]:
np.random.seed(0)
rho = 0.4
sig = np.sqrt(0.036**2*(1-0.4**2))
mu = 0.018
gamma = 10
beta = 0.99
n=100000

log_consumption_growth = np.zeros(n)
epsilon = np.random.normal(0,1,n)
for i in range(len(log_consumption_growth)):
    if i == 0:
        log_consumption_growth[i] = (1-rho)*mu + sig*epsilon[i]
    else:
        log_consumption_growth[i] = (1-rho)*mu + rho *log_consumption_growth[i-1] + sig*epsilon[i]

sdf = np.log(beta) -gamma*log_consumption_growth
rf = -np.log(beta) + gamma*(1-rho)*mu + gamma*rho*log_consumption_growth - gamma**2*sig**2/2
rf_bar =  -np.log(beta) + gamma * mu  -0.5 * gamma**2 * sig**2

In [49]:
from scipy.optimize import brentq

#calculate dp_bar for rh_cs in campbell shiller
def gap(dp):
    rho_cs = 1/ (1+np.exp(dp))
    kappa = -np.log(rho_cs) -(1-rho_cs)*np.log(1/rho_cs-1)
    b1 = (gamma-1)* rho /(1-rho_cs*rho)
    risk_prem = gamma*sig**2 * (1-rho_cs*b1)
    b0= (-kappa-mu+rf_bar+risk_prem)/(1-rho_cs)
    return b0-dp

dp_bar = brentq(gap,-10,10)

rho_cs = 1/ (1+np.exp(dp_bar))
kappa = -np.log(rho_cs) -(1-rho_cs)*np.log(1/rho_cs-1)
b1 = (gamma-1)* rho /(1-rho_cs*rho)
risk_prem = gamma*sig**2 * (1-rho_cs*b1)
b0= (-kappa-mu + rf_bar + risk_prem)/(1-rho_cs)

log_ret = kappa + log_consumption_growth[1:] + (1-rho_cs)*b0 - rho_cs*b1*(log_consumption_growth[1:]-mu) +b1*(log_consumption_growth[:-1]-mu)

risk_prem = log_ret - rf[:-1]


In [50]:
print('Means')
print(f'Log risk-free rate:     {np.round(np.mean(rf), 4)}')
print(f'Log consumption growth: {np.round(np.mean(log_consumption_growth), 4)}')
print(f'Log SDF:                {np.round(np.mean(sdf), 4)}')
print(f'Log equity return:      {np.round(np.mean(log_ret), 4)}')
print(f'Log excess return:      {np.round(np.mean(risk_prem), 4)}')

print('')
print('Unconditional Standard Deviations')
print(f'Log risk-free rate:     {np.round(np.std(rf), 4)}')
print(f'Log consumption growth: {np.round(np.std(log_consumption_growth), 4)}')
print(f'Log SDF:                {np.round(np.std(sdf), 4)}')
print(f'Log equity return:      {np.round(np.std(log_ret), 4)}')
print(f'Log excess return:      {np.round(np.std(risk_prem), 4)}')

print('')
print('First Autocorrelations')
print(f'Log risk-free rate:     {np.round(acf(rf, nlags=1)[1], 4)}')
print(f'Log consumption growth: {np.round(acf(log_consumption_growth, nlags=1)[1], 4)}')
print(f'Log SDF:                {np.round(acf(sdf, nlags=1)[1], 4)}')
print(f'Log equity return:      {np.round(acf(log_ret, nlags=1)[1], 4)}')
print(f'Log excess return:      {np.round(acf(risk_prem, nlags=1)[1], 4)}')

print('')
print(f'Correlation of consumption growth and equity return {np.corrcoef(log_consumption_growth[1:], log_ret)[0][1]}')

Means
Log risk-free rate:     0.136
Log consumption growth: 0.0181
Log SDF:                -0.1909
Log equity return:      0.0884
Log excess return:      -0.0476

Unconditional Standard Deviations
Log risk-free rate:     0.1437
Log consumption growth: 0.0359
Log SDF:                0.3592
Log equity return:      0.2027
Log excess return:      0.1432

First Autocorrelations
Log risk-free rate:     0.401
Log consumption growth: 0.401
Log SDF:                0.401
Log equity return:      -0.2548
Log excess return:      0.0024

Correlation of consumption growth and equity return -0.3630542770365999


Adding persistence increased the log risk free rate mean slightly and is more volatile from consumption growth. Log equity return mean decreased and is more volatile. SDF and log consumption growth stay the same in terms of mean and variance. Also the auto correlations for risk free and sdf increase given then autocorrelation of log consumption growth increases. Log equity return now has negative auto correlation and excess still has 0 auto correlation. We also see negative correlation with returns and consumption meaning the equity does well in bad times. Thus the return is lower despite higher volatility given that it acts as insurance. The autocorrelation dynamic adds mean reversion to the asset and makes the risk free rate volatile. 

### Problem 2(c)

In [51]:
n=5
A = np.zeros(n+1)
B = np.zeros(n+1)
for i in range(1,n+1):
    B[i] = 1 + rho*B[i-1]

    delta = 0.5*gamma**2*sig**2
    new_sig = gamma*rho*sig
    theta = rf_bar

    A[i] = A[i-1] + delta + B[i-1]*(1-rho)*theta - (gamma*sig + B[i-1]*new_sig)**2/2

prices = np.empty(shape=(len(rf),n+1))
for i in range(len(rf)):
    prices[i] = -A -B*rf[i]

In [52]:
from statsmodels.tsa.stattools import acf

print('Mean Yields')
for i in range(1,6):
    y = -prices[:, i] / i
    print(f'{i} year: {np.round(np.mean(y), 4)}')

print('')
print('Unconditional Standard Deviations')
for i in range(1, 6):
    y = -prices[:, i] / i
    print(f'{i} year: {np.round(np.std(y), 4)}')

print('')
print('First Autocorrelations')
for i in range(1, 6):
    y = -prices[:, i] / i
    print(f'{i} year: {np.round(acf(y, nlags=1)[1], 4)}')


Mean Yields
1 year: 0.136
2 year: 0.1097
3 year: 0.0924
4 year: 0.0809
5 year: 0.0731

Unconditional Standard Deviations
1 year: 0.1437
2 year: 0.1006
3 year: 0.0747
4 year: 0.0583
5 year: 0.0474

First Autocorrelations
1 year: 0.401
2 year: 0.401
3 year: 0.401
4 year: 0.401
5 year: 0.401


In [53]:
rhos =  np.linspace(0.0,1.0,5)
A_vec = np.empty(shape=(len(rhos),6))
B_vec = np.empty(shape=(len(rhos),6))
y_vec = np.zeros(shape=(len(rhos),6))

j = 0
for rho in rhos:
    n=5
    A = np.zeros(n+1)
    B = np.zeros(n+1)
    for i in range(1,n+1):
        B[i] = 1 + rho*B[i-1]

        delta = 0.5*gamma**2*sig**2
        new_sig = gamma*rho*sig
        theta = rf_bar

        A[i] = A[i-1] + delta + B[i-1]*(1-rho)*theta - (gamma*sig + B[i-1]*new_sig)**2/2
    A_vec[j,:] = A
    B_vec[j,:] = B

    price = -A-B*rf_bar
    y_vec[j,1:] = -price[1:]/np.arange(1,n+1)
    j+=1

print(A_vec.round(2))
print(B_vec.round(2))
print(y_vec.round(2))

[[ 0.    0.    0.14  0.27  0.41  0.54]
 [ 0.    0.    0.07  0.16  0.25  0.34]
 [ 0.    0.   -0.   -0.01 -0.03 -0.05]
 [ 0.    0.   -0.08 -0.26 -0.53 -0.89]
 [ 0.    0.   -0.16 -0.6  -1.42 -2.72]]
[[0.   1.   1.   1.   1.   1.  ]
 [0.   1.   1.25 1.31 1.33 1.33]
 [0.   1.   1.5  1.75 1.88 1.94]
 [0.   1.   1.75 2.31 2.73 3.05]
 [0.   1.   2.   3.   4.   5.  ]]
[[ 0.    0.14  0.14  0.14  0.14  0.14]
 [ 0.    0.14  0.12  0.11  0.11  0.1 ]
 [ 0.    0.14  0.1   0.08  0.06  0.04]
 [ 0.    0.14  0.08  0.02 -0.04 -0.1 ]
 [ 0.    0.14  0.05 -0.06 -0.22 -0.41]]


if we increase rho then B increases for next year yield. This means the yield increases at a faster rate from maturity year to year. I rho increases then A decreases, thus the initial yield is lower and A doesnt increase as quickly and can even be negative for older maturities. So as rho increases shorter maturities years are increasing, but the older yields tend to fall if rho is high enough, like 0.8.